# 11. Three-phase houses with a single-phase inverter (3L1P): Volt-VAr validation

**Question.** On a three-phase house with one single-phase inverter, does the single-phase blind Volt-VAr test (notebooks 06/07) still work when it is applied to **one** house phase of the net meter?

**Method: identical to 06/07.** 
- Monthly 00-04 h AEST night baseline; 
- `P_est = base − net`, `Q_est = base − net`;
- keep intervals with 242 ≤ V < 253 V, `P_est ≥ 0.20 × rating` (signed) and `Q_est` present; 
- `q = 100 Q_est / rating`;
- OLS of q on V; 
- assessable if n ≥ 20 and ΔV ≥ 5 V; 
- conformant if the slope is in [−4.000, −2.667] %/V.
- The rating is the site rating: the inverter is single-phase, so all its output is on one phase.
- The reference is the PV circuit's own Q-V slope with the same window, gate (|P|) and band; 
- PV circuits reporting a constant Q are not assessable (as 07).

**Only the tested circuit changes:**

| Route | Circuit tested | Uses |
|---|---|---|
| Grount truth | the house phase the PV circuit is on (voltage matching, notebook 10) | answer key |
| Blind route | the phase the blind classifier picked, in months it decided `1P` | net meter only |

**Populations** (all 3L1P, reference configuration `1P`):

| | Sites | Used for |
|---|---|---|
| A | clean (`use_in_11`, ≥ 2 decided months) | main result |
| A3 | A with `n_decided` ≥ 3 | stricter subset |
| B | every reference-graded 1P site | end-to-end blind route |
| C | excluded in 10 only for blind reasons, with a strong voltage reference | selection-bias check |

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bms_sa_review.ami_analysis.lib import ami_fleet_vvar as Fleet
from bms_sa_review.ami_analysis.lib import ami_phase_vvar as Phase
from bms_sa_review.ami_analysis.lib import ami_vvar_validation as Valid
from bms_sa_review.ami_analysis.lib import ami_phase1p_vvar as P1

## 1. Settings

In [ ]:
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]

# Method settings: identical to notebooks 06/07
NIGHT_HOURS = (0, 4)
P_MIN_FRAC = Fleet.P_MIN_FRAC          # signed gate on the net meter, |P| gate on the PV circuit
V_LO, V_HI = 242, 253
FIT_SETTINGS = dict(night_hours=NIGHT_HOURS, p_min_frac=P_MIN_FRAC, v_lo=V_LO, v_hi=V_HI)
MIN_N_WINDOW, MIN_V_SPREAD = 20, 5.0
SLOPE_TOLERANCE_PCT = 20               # band [-4.000, -2.667] %/V

# Notebook 11 settings
REF_DV_RATIO_MIN = 5.0                 # strong voltage reference, as notebook 10 (population C)
LOW_PV_MAX = 0.45                      # low-output months: PV circuit midday q90 / rating below this
N_BOOT, SEED = 1000, 42                # site-cluster bootstrap
N_1L1P_SAMPLE = 3000                   # single-phase site-months sampled for the beta_e comparison
MIN_INCOMPLETE_MONTHS = 6              # "affected" by the missing paired circuit: >= this many incomplete months in a year
SAMPLE_N_SITES = None                  # e.g. 30 for a quick test run (seeded random sites); None = all sites

DUCKDB_MEMORY, DUCKDB_THREADS = "8GB", 4
OVERWRITE = False

PHASE_DIR = Fleet.store_dir() / "vvar_phase" / "phase_config"     # notebook 10 outputs
FLEET_DIR = Fleet.store_dir() / "vvar_fleet" / f"2024_2025_band_{V_LO}_{V_HI}"   # notebook 07 outputs (1L1P)
OUT_DIR = Fleet.store_dir() / "vvar_phase" / "3l1p_vvar"
OUT_DIR.mkdir(parents=True, exist_ok=True)
KEYS = P1.KEYS

lo, hi = Fleet.REQUIRED_SLOPE * (1 + SLOPE_TOLERANCE_PCT / 100), Fleet.REQUIRED_SLOPE * (1 - SLOPE_TOLERANCE_PCT / 100)
print(f"Band: [{lo:.3f}, {hi:.3f}] %/V | window [{V_LO}, {V_HI}) V | outputs -> {OUT_DIR}")

## 2. Inputs and populations

From notebook 10: 
- `site_evidence` (one row per site), 
- `blind_config_site_month` (blind result per site-month, with `month_ok`) and `truth_pairing_site_month` (voltage match per site-month). 
- Population C applies the phase-reference check explicitly, because notebook 10 stops at the first failed check (a `too_few_decided_months` site was never tested for the strength of its voltage reference).

In [ ]:
ev = pd.read_parquet(PHASE_DIR / "site_evidence.parquet")
blind_month = pd.read_parquet(PHASE_DIR / "blind_config_site_month.parquet")
truth_pairing = pd.read_parquet(PHASE_DIR / "truth_pairing_site_month.parquet")
ref_month = P1.reference_phase_month(truth_pairing)

pop = P1.populations(ev, ref_dv_ratio_min=REF_DV_RATIO_MIN)
sites = P1.fit_sites(ev)
if SAMPLE_N_SITES:
    sites = sites.sample(n=min(SAMPLE_N_SITES, len(sites)), random_state=SEED).reset_index(drop=True)
    pop = pop[pop.site_id.isin(sites.site_id)].reset_index(drop=True)
    print(f"TEST RUN: {len(sites)} random sites")

display(pop[["in_B", "in_A", "in_A3", "in_C"]].sum().rename("sites").to_frame())
print("Sites excluded in notebook 10 only for blind reasons, by strength of the voltage reference:")
display(pd.crosstab(pop[pop.blind_only_excluded].exclusion_reason, pop[pop.blind_only_excluded].ref_strong,
                    margins=True).rename(columns={True: "strong (-> C)", False: "weak"}))

ok = blind_month[blind_month.month_ok & blind_month.site_id.isin(pop[pop.in_A].site_id)]
print("month_ok site-months of A, by year:", ok.groupby("year").size().to_dict(), "| total", len(ok))

## 3. Fits (cached per month)

`Fleet.run_months` fits every circuit of a site, so the net-meter run gives one fit per house phase and the
reference run one fit for the PV circuit. Months on disk are skipped; the site list is checked so a cache built for
another sample is never reused silently.

In [ ]:
P1.check_site_list(OUT_DIR, sites, OVERWRITE)
for name, source in [("meter", "meter_baseline"), ("truth", "ground_truth")]:
    log = Fleet.run_months(MONTHS, OUT_DIR / name, sites, source=source, overwrite=OVERWRITE,
                           memory_limit=DUCKDB_MEMORY, threads=DUCKDB_THREADS, **FIT_SETTINGS)

meter, truth = P1.load_fits(OUT_DIR, sites, min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD,
                            tolerance_pct=SLOPE_TOLERANCE_PCT, v_lo=V_LO, v_hi=V_HI)
in_months = lambda d: d[(d.year.astype(str) + "-" + d.month.map("{:02d}".format)).isin(MONTHS)]
meter, truth = in_months(meter), in_months(truth)

print(f"Net-meter fits: {len(meter):,} circuit-months; reference fits: {len(truth):,} site-months")
display(pd.crosstab(truth.year, truth.fit_status, margins=True))

## 4. Oracle validation (population A, `month_ok` months)

- The net-meter circuit tested is `ref_phase`, the house phase the PV circuit is on. 
- Agreement and F1 are against the PV circuit's verdict (positive class = conformant), on site-months where both are assessable. 
- The single-phase result (notebook 06, 1L1P) is shown alongside.

In [ ]:
orc = P1.oracle_table(meter, truth, pop, blind_month, ref_month)
orc["slope_err"] = orc.slope_blind - orc.slope_truth
A = orc[orc.in_A & orc.month_ok]
main = A[A.both_ok].copy()
main["model_hint"] = main.model_hint.fillna("none")

print(f"A, month_ok site-months with a reference row: {len(A):,}")
display(pd.DataFrame({"net meter (ref phase)": A.fit_status_oracle.value_counts(),
                      "PV circuit": A.fit_status_truth.value_counts()}).fillna(0).astype(int))
print(f"Both assessable: {len(main):,} site-months at {main.site_id.nunique():,} sites")

cols = ["sites", "n", "TP", "FP", "FN", "TN", "accuracy", "precision", "recall", "specificity", "f1",
        "pass_rate_tested_%", "pass_rate_reference_%"]
oracle_metrics = pd.concat([P1.metrics(main), P1.metrics(main, "year"), P1.metrics(main, "model_hint")],
                           keys=["all", "year", "model hint"])
display(oracle_metrics[cols].round(3))

compare = pd.DataFrame({
    "3L1P oracle accuracy": [oracle_metrics.loc[("all", "all"), "accuracy"]]
                            + [oracle_metrics.loc[("year", y), "accuracy"] if ("year", y) in oracle_metrics.index else np.nan for y in (2024, 2025)],
    "3L1P oracle F1": [oracle_metrics.loc[("all", "all"), "f1"]]
                      + [oracle_metrics.loc[("year", y), "f1"] if ("year", y) in oracle_metrics.index else np.nan for y in (2024, 2025)],
    "1L1P accuracy (06/09)": P1.SINGLE_PHASE_REFERENCE.accuracy.values,
    "1L1P F1 (06/09)": P1.SINGLE_PHASE_REFERENCE.f1.values,
}, index=["all years", "2024", "2025"])
display(compare.round(3))

In [ ]:
fig = Valid.plot_confusion_matrices(main, pairs=[("blind", "truth")])
fig.axes[0].set_ylabel("Net meter, reference phase (oracle)", fontsize=9)
fig.axes[0].set_xlabel("PV circuit (reference)", fontsize=9)
fig.savefig(OUT_DIR / "oracle_confusion.png", bbox_inches="tight")
plt.show()

Fleet.plot_blind_vs_truth(main)
plt.show()

## 5. Main result vs the stricter subset

Main = A (≥ 2 decided months); sensitivity = A3 (`n_decided` ≥ 3). 95% confidence intervals from a site-cluster
bootstrap (sites resampled with replacement). **Rule:** if the A3 accuracy and F1 both fall inside the main result's
intervals, the 2-month result is the main one; otherwise the 3-month result is used.

In [ ]:
ci = pd.DataFrame({
    "main (n_decided >= 2)": P1.bootstrap_ci(main, N_BOOT, SEED),
    "sensitivity (n_decided >= 3)": P1.bootstrap_ci(main[main.in_A3], N_BOOT, SEED),
}).T
display(ci.round(4))

m, s = ci.iloc[0], ci.iloc[1]
close = (m.accuracy_lo <= s.accuracy <= m.accuracy_hi) and (m.f1_lo <= s.f1 <= m.f1_hi)
MAIN_LABEL = "main (n_decided >= 2)" if close else "sensitivity (n_decided >= 3)"
main_set = main if close else main[main.in_A3]
print(f"A3 inside the main intervals: {close} -> reported result: {MAIN_LABEL}")

by_year = pd.concat({"main": P1.metrics(main, "year"), "n_decided >= 3": P1.metrics(main[main.in_A3], "year")}, axis=1)
display(by_year.loc[:, (slice(None), ["sites", "n", "accuracy", "f1"])].round(3))

## 6. End-to-end blind route (population B)

Every reference-assessable site-month of every reference-1P site, routed by the net meter only:

1. **Configuration**: that month's blind result. 
`1P` → single-phase test; 
`3P` → wrong route, **counted as an error**;
`incomplete` / `no_signal` / `unclear` / no signature → not assessable.
2. **Phase**: that month's blind phase, graded against the phase the PV circuit is confidently matched to that month (else the site's consensus phase when its reference is strong).
3. **Volt-VAr**: the single-phase test on the routed circuit vs the PV circuit.

The last two rows of the summary compare the blind route and the oracle on the same months: the difference is the
cost of the routing.

In [ ]:
e2e = P1.route_end_to_end(meter, truth, pop, blind_month, ref_month, phase_choice="month")
display(P1.step_table(e2e))
e2e_summary = P1.end_to_end_summary(e2e)
display(e2e_summary)

print("By year:")
display(pd.concat({y: P1.end_to_end_summary(g)[["site_months", "accuracy"]] for y, g in e2e.groupby("year")}, axis=1))
print("Clean (A) vs other reference-1P sites:")
display(pd.concat({"A": P1.end_to_end_summary(e2e[e2e.in_A])[["site_months", "accuracy"]],
                   "not A": P1.end_to_end_summary(e2e[~e2e.in_A])[["site_months", "accuracy"]]}, axis=1))

routed = e2e[e2e.outcome.isin(["agree", "disagree"])]
print("Given a 1P route (positive = conformant):")
display(P1.metrics(routed, pred="passed_route")[cols].round(3))

In [ ]:
# Sensitivity: route by the site's majority blind phase instead of each month's blind phase
e2e_site = P1.route_end_to_end(meter, truth, pop, blind_month, ref_month, phase_choice="site")
display(pd.concat({"month phase (main)": e2e_summary[["site_months", "accuracy"]],
                   "site-vote phase": P1.end_to_end_summary(e2e_site)[["site_months", "accuracy"]]}, axis=1))

## Daily plots

In [ ]:
# False negatives: net meter fails, PV circuit passes. One figure per site-month.
from bms_sa_review.ami_analysis.lib import ami_phase1p_plots as P1Plots

display(P1Plots.outcome_overview(main))

cases = P1Plots.select_cases(main, outcome="FN", n=6, how="random")   # how="worst" / "borderline"; n=None = all 95
display(cases.round(3))
for _, case in cases.iterrows():
    fig, data = P1Plots.plot_case(case)        # e.g. plot_case(case, date="2025-03-17") for a specific AEST day
    plt.show()

In [ ]:
# False positives: net meter passes, PV circuit fails. One figure per site-month.
cases_fp = P1Plots.select_cases(main, outcome="FP", n=6, how="random")   # n=None = all 41
display(cases_fp.round(3))
for _, case in cases_fp.iterrows():
    fig, data = P1Plots.plot_case(case)
    plt.show()

## Interactive plots

In [ ]:
from bms_sa_review.ami_analysis.lib import interactive_plots as IPlots

cache = IPlots.explore_1p(main, ev=ev)                         # starts on FN, worst first
# cache = IPlots.explore_1p(main, ev=ev, outcome="FP", how="borderline")

## Phantom export check

The night baseline is the monthly 00-04 h median. If a regular night load (e.g. off-peak water heating) is part of
that median, then whenever it is off, `base − net` looks like export, can pass the 20% gate and enters the fit with no
PV behind it (example in the FN plots: site 936715918, July 2024).

For each oracle site-month (section 4): the share of the net-meter fit's intervals in which the PV circuit's |P| is
below `PV_OFF_FRAC` of rating (no PV output), and, as a version that needs no reference, the share outside daylight
(AEST hour < `DAY_START` or ≥ `DAY_END`). Diagnostic only: the method is unchanged.

In [ ]:
PV_OFF_FRAC, DAY_START, DAY_END = 0.05, 7, 18
units_ph = main[KEYS + ["ref_phase", "rating_kw_oracle"]].rename(
    columns={"ref_phase": "meter_circuit_id", "rating_kw_oracle": "rating_kw"})
phantom = P1.run_outside_daylight(units_ph, MONTHS, OUT_DIR / "phantom",
                                  f"phantom_3l1p_oracle_pv{PV_OFF_FRAC:g}_day{DAY_START}-{DAY_END}",
                                  memory_limit=DUCKDB_MEMORY, threads=DUCKDB_THREADS, overwrite=OVERWRITE,
                                  night_hours=NIGHT_HOURS, p_min_frac=P_MIN_FRAC, v_lo=V_LO, v_hi=V_HI,
                                  day_start=DAY_START, day_end=DAY_END, pv_off_frac=PV_OFF_FRAC)
ph = P1Plots.add_outcome(main).merge(phantom, on=KEYS)
assert len(ph) == len(main) and (ph.n_window_fit == ph.n_window_oracle).all(), "interval counts differ from the fits"
ph["share_pv_off"] = ph.n_pv_off / ph.n_window_fit
ph["share_outside_daylight"] = ph.n_outside_daylight / ph.n_window_fit

print(f"Share of fitted net-meter intervals with no PV output (|P_pv| < {PV_OFF_FRAC:.0%} of rating), by outcome:")
phantom_table = P1.outside_daylight_table(ph, "share_pv_off")
display(phantom_table.round(3))
print(f"Reference-free version: share outside daylight (before {DAY_START}:00 or from {DAY_END}:00 AEST):")
display(P1.outside_daylight_table(ph, "share_outside_daylight").round(3))
display(pd.crosstab(ph.share_pv_off > 0.10, ph.share_outside_daylight > 0.10,
                    rownames=["no PV output > 10%"], colnames=["outside daylight > 10%"]))

ph["phantom_group"] = np.where(ph.share_pv_off > 0.10, "> 10% of fitted intervals with no PV output", "<= 10%")
display(P1.metrics(ph, "phantom_group")[["sites", "n", "TP", "FP", "FN", "TN", "accuracy", "f1"]].round(3))
phantom_table.to_csv(OUT_DIR / "phantom_export.csv")

## 7. Selection-bias check

**Population C**: sites excluded in notebook 10 only for blind reasons, with a strong voltage reference. Their months are
filtered with a **reference-only** rule (PV circuit confidently matched to `ref_phase` that month), because
`month_ok` needs the blind result. A is shown under both filters so the comparison is like for like. If C's agreement
falls well below A's, the clean selection makes the result look better than it is.

In [ ]:
groups = {
    "A, month_ok (main)": orc[orc.in_A & orc.month_ok & orc.both_ok],
    "A, reference-only month filter": orc[orc.in_A & orc.ref_month_ok & orc.both_ok],
    "C, reference-only month filter": orc[orc.in_C & orc.ref_month_ok & orc.both_ok],
}
bias = pd.DataFrame({k: P1.bootstrap_ci(g, N_BOOT, SEED) if len(g) else {} for k, g in groups.items()}).T
display(bias.round(4))
c_only = groups["C, reference-only month filter"]
if len(c_only):
    print("C by exclusion reason:")
    display(P1.metrics(c_only, "exclusion_reason")[["sites", "n", "accuracy", "f1"]].round(3))

In [ ]:
# Low PV output months: PV circuit midday q90 / rating < LOW_PV_MAX
pvr = P1.pv_output_ratio(PHASE_DIR / "pv_output", pop)
A_all = A.merge(pvr, on=KEYS, how="left")
print(f"A month_ok site-months with q90/rating < {LOW_PV_MAX}: {(A_all.pv_q90_over_rating < LOW_PV_MAX).mean():.1%} "
      f"of {len(A_all):,} (no PV output record: {A_all.pv_q90_over_rating.isna().sum()})")
mo = main.merge(pvr, on=KEYS, how="left")
mo["pv_output"] = np.where(mo.pv_q90_over_rating < LOW_PV_MAX, f"low (< {LOW_PV_MAX})", "normal")
display(P1.metrics(mo, "pv_output")[["sites", "n", "TP", "FP", "FN", "TN", "accuracy", "f1"]].round(3))

## 8. Missing paired-phase circuit

Some clean 1P sites lose the house-phase circuit the PV is on for a whole year, so those months are `incomplete` and
`month_ok` drops them. Checks:

1. Which table loses it first: `ami_extract` (the Athena extract), `ami_raw_phaseseparate`, `ami_meter_residential`.
2. Bias: the affected sites still have their other year. Their oracle result in that year is compared with the
   unaffected sites in the same year.

In [ ]:
aff = P1.affected_sites(blind_month, MIN_INCOMPLETE_MONTHS)
aff = aff[aff.site_id.isin(pop.site_id)]
print("Clean 1P sites by affected year (0 = not affected):", aff.affected_year.value_counts().to_dict())
inc = blind_month[(blind_month.exclusion_reason == "clean") & (blind_month.expected == "1P")
                  & (blind_month.blind_config == "incomplete") & blind_month.site_id.isin(pop.site_id)]
print(f"Incomplete site-months of clean 1P sites: {len(inc):,}")

# Presence of the paired circuit, month by month, in each table (cached; ami_extract: load files only)
circ = (pop[pop.site_id.isin(aff[aff.affected_year > 0].site_id)][["site_id", "ref_phase"]]
        .rename(columns={"ref_phase": "circuit_id"}).astype({"circuit_id": "int64"}))
pres_path = OUT_DIR / "missing_circuit_presence.parquet"
if pres_path.exists() and not OVERWRITE:
    presence = pd.read_parquet(pres_path)
else:
    parts = []
    for month in MONTHS:
        for table, glob in [("ami_extract", "part-load-*.parquet"), ("ami_raw_phaseseparate", "*.parquet"),
                            ("ami_meter_residential", "*.parquet")]:
            if not Fleet.month_dir(table, month).exists():
                continue
            con = Fleet.connect(DUCKDB_MEMORY, DUCKDB_THREADS)
            try:
                con.register("_circ", circ[["circuit_id"]])
                parts.append(con.execute(P1.presence_sql(table, month, glob)).df())
            finally:
                con.close()
    presence = pd.concat(parts, ignore_index=True)
    presence.to_parquet(pres_path, index=False)

presence = presence.merge(circ.merge(aff[["site_id", "affected_year"]], on="site_id"), on="circuit_id")
grid = (presence.pivot_table(index="ym", columns=["affected_year", "tbl"], values="circuit_id", aggfunc="nunique")
        .fillna(0).astype(int))
print("Affected sites by the year they lose the circuit:",
      aff[aff.affected_year > 0].affected_year.value_counts().to_dict())
print("Paired circuits present, by month, affected year and table:")
display(grid)

In [ ]:
# Proposed Athena check (not run here): does the Solar Analytics source have rows for these circuits?
print(P1.athena_presence_sql(circ.circuit_id))

In [ ]:
# Bias check: oracle result of affected sites in their intact year vs unaffected sites in the same year
mb = main.merge(aff[["site_id", "affected_year"]], on="site_id", how="left")
mb["affected_year"] = mb.affected_year.fillna(0).astype(int)
rows = {}
for intact_year, lost_year in [(2024, 2025), (2025, 2024)]:
    g = mb[mb.year == intact_year]
    rows[(intact_year, f"affected (lost {lost_year})")] = P1.metrics(g[g.affected_year == lost_year]).iloc[0]
    rows[(intact_year, "not affected")] = P1.metrics(g[g.affected_year == 0]).iloc[0]
display(pd.DataFrame(rows).T[["sites", "n", "accuracy", "f1", "pass_rate_tested_%", "pass_rate_reference_%"]].round(3))

## 9. Other-phase effect

On a 3L1P house the tested phase carries only part of the household load; the other two phases carry the rest and
no inverter. Two checks against the single-phase cohort (1L1P, notebook 07):

1. **Night baseline** (`base_p`, `base_q`, % of rating): paired phase, the other two phases, and the 1L1P circuit.
2. **Household reactive power on the tested circuit** (notebook 09): on intervals in both fit windows,
   `e = Q_est − Q_PV` (% of rating), `β_e = cov(V, e) / var(V)` and sd(e). A random sample of 1L1P site-months with
   both fits assessable is used for comparison.

In [ ]:
fleet = pd.read_parquet(FLEET_DIR / "site_month_results.parquet")
fleet = in_months(fleet)
fb, ft = fleet[fleet.source == "meter_baseline"], fleet[fleet.source == "ground_truth"]
one = fb[fb.fit_status == "ok"].merge(ft.loc[ft.fit_status == "ok", KEYS + ["slope"]], on=KEYS,
                                      suffixes=("", "_truth"))   # 1L1P, both assessable
one["slope_err"] = one.slope - one.slope_truth

pct = lambda d, r: d.assign(base_p_pct=100 * d.base_p_kw / d[r], base_q_pct=100 * d.base_q_kvar / d[r])
paired = pct(main.rename(columns={"base_p_kw_oracle": "base_p_kw", "base_q_kvar_oracle": "base_q_kvar"}), "rating_kw_oracle")
others = meter.merge(main[KEYS + ["ref_phase"]], on=KEYS)
others = pct(others[(others.circuit_id != others.ref_phase).fillna(False)], "rating_kw")
single = pct(one, "rating_kw")
base_cmp = P1.distribution_table({"1L1P circuit (whole house)": single, "3L1P paired phase": paired,
                                  "3L1P other two phases": others}, ["base_p_pct", "base_q_pct"])
display(base_cmp.round(3))

In [ ]:
units3 = main[KEYS + ["ref_phase", "rating_kw_oracle", "slope_err"]].rename(
    columns={"ref_phase": "meter_circuit_id", "rating_kw_oracle": "rating_kw"})
units1 = (one.sample(n=min(N_1L1P_SAMPLE, len(one)), random_state=SEED)
          [KEYS + ["circuit_id", "rating_kw", "slope_err"]].rename(columns={"circuit_id": "meter_circuit_id"}))

beta3 = P1.run_beta_e(units3, MONTHS, OUT_DIR / "beta_e", "3l1p_oracle", memory_limit=DUCKDB_MEMORY,
                      threads=DUCKDB_THREADS, overwrite=OVERWRITE, night_hours=NIGHT_HOURS,
                      p_min_frac=P_MIN_FRAC, v_lo=V_LO, v_hi=V_HI)
beta1 = P1.run_beta_e(units1, MONTHS, OUT_DIR / "beta_e", f"1l1p_sample_n{N_1L1P_SAMPLE}_seed{SEED}",
                      memory_limit=DUCKDB_MEMORY, threads=DUCKDB_THREADS, overwrite=OVERWRITE,
                      night_hours=NIGHT_HOURS, p_min_frac=P_MIN_FRAC, v_lo=V_LO, v_hi=V_HI)
beta3 = beta3.merge(units3[KEYS + ["slope_err"]], on=KEYS)
beta1 = beta1.merge(units1[KEYS + ["slope_err"]], on=KEYS)

display(P1.distribution_table({"1L1P sample": beta1, "3L1P paired phase": beta3},
                              ["beta_e", "e_sd", "slope_err", "n_shared"]).round(3))
for name, b in [("1L1P sample", beta1), ("3L1P paired phase", beta3)]:
    print(f"{name}: corr(beta_e, slope error) = {b.beta_e.corr(b.slope_err):.3f} (n = {len(b):,})")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), dpi=120)
for (name, b), c in zip([("1L1P sample", beta1), ("3L1P paired phase", beta3)], ["#888780", "#D85A30"]):
    axes[0].hist(b.beta_e.clip(-3, 3), bins=80, density=True, histtype="step", color=c, label=name)
    axes[1].hist(b.e_sd.clip(upper=30), bins=80, density=True, histtype="step", color=c, label=name)
axes[0].set(xlabel="β_e (%/V, clipped at ±3)", ylabel="Density", title="Household reactive power slope term")
axes[1].set(xlabel="sd(e) (% of rating, clipped at 30)", title="Spread of household reactive power")
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "other_phase_beta_e.png", bbox_inches="tight")
plt.show()

## 10. Outputs

| File | Content |
|---|---|
| `site_month_results_3l1p.parquet` | `Phase.RESULT_COLUMNS`; `source` = `meter_oracle` (reference phase), `meter_blind_route` (blind phase, months the blind method decided `1P`), `ground_truth` (PV circuit); `unit_id = site_id` |
| `site_month_details_3l1p.parquet` | per site-month: populations, `month_ok`, `ref_month_ok`, tested circuits, end-to-end outcome, PV output ratio, affected year |
| `*.csv` | metric tables from sections 4-7 |

Notebook 13 should use `meter_blind_route` for the fleet result and the details file for filters.

In [ ]:
# Full fit rows for each route (Phase.RESULT_COLUMNS needs v_min/v_max/stderr etc.)
def fit_rows(units, circuit_col):
    u = units[KEYS + [circuit_col]].dropna().astype({circuit_col: "int64"}).drop_duplicates(KEYS)
    m = meter.astype({"circuit_id": "Int64"})
    return u.merge(m, left_on=KEYS + [circuit_col], right_on=KEYS + ["circuit_id"], how="left")

oracle_units = orc[KEYS + ["ref_phase"]]
route_units = (blind_month[blind_month.site_id.isin(pop.site_id) & (blind_month.blind_config == "1P")]
               [KEYS + ["blind_load_circuit_id"]])
route_units = in_months(route_units)
results = pd.concat([
    P1.to_result_columns(fit_rows(oracle_units, "ref_phase").fillna({"fit_status": "no_circuit_data"}), source="meter_oracle"),
    P1.to_result_columns(fit_rows(route_units, "blind_load_circuit_id").fillna({"fit_status": "no_circuit_data"}),
                         source="meter_blind_route"),
    P1.to_result_columns(truth[truth.site_id.isin(pop.site_id)], source="ground_truth"),
], ignore_index=True)
assert not results.duplicated(["site_id", "year", "month", "source"]).any()
results.to_parquet(OUT_DIR / "site_month_results_3l1p.parquet", index=False)

details = (orc[KEYS + ["in_A", "in_A3", "in_C", "month_ok", "ref_month_ok", "ref_phase", "ref_phase_month",
                       "blind_config", "exclusion_reason", "model_hint"]]
           .merge(e2e[KEYS + ["route", "route_circuit", "phase_status", "outcome"]], on=KEYS, how="left")
           .merge(pvr, on=KEYS, how="left")
           .merge(aff[["site_id", "affected_year"]], on="site_id", how="left"))
details.to_parquet(OUT_DIR / "site_month_details_3l1p.parquet", index=False)

oracle_metrics.to_csv(OUT_DIR / "oracle_metrics.csv")
ci.to_csv(OUT_DIR / "main_vs_strict.csv")
P1.step_table(e2e).to_csv(OUT_DIR / "end_to_end_steps.csv")
e2e_summary.to_csv(OUT_DIR / "end_to_end_summary.csv")
bias.to_csv(OUT_DIR / "selection_bias.csv")
base_cmp.to_csv(OUT_DIR / "other_phase_baseline.csv")
display(results.groupby("source").fit_status.value_counts().unstack(fill_value=0))
print(f"Saved to {OUT_DIR}")

## 11. Reading the results (full run, 6 Nov 2026)

- **Populations (2).** B 598, A 479, A3 460, C 44 sites. A has 7,244 `month_ok` site-months (4,220 in 2024, 3,024 in 2025).
- **Oracle (4).** Both verdicts assessable in 4,564 site-months at 389 sites (63% of `month_ok`). Accuracy 0.970, F1 0.908
  (TP 675, FP 41, FN 95, TN 3,753); 2024: 0.979 / 0.926; 2025: 0.958 / 0.890. Single-phase (1L1P, notebook 06): 0.945 / 0.873.
  The sites, months and selection differ, so this shows the test carries over to one phase of a three-phase meter once the
  phase is known; it does not show that it works better.
- **Main vs stricter (5).** A3: 0.970 / 0.908 on 4,541 site-months, inside A's 95% intervals (accuracy 0.957-0.982,
  F1 0.865-0.940), so the 2-month result is the main one.
- **End-to-end (6).** 6,103 reference-assessable site-months: 834 undecided (13.7%), 26 routed 3P (0.4%), 5,243 routed 1P
  (phase correct 4,839, wrong 100, no reference 304). Accuracy 0.956 at 83.8% coverage; 0.961 given a 1P route (F1 0.884).
  A sites 0.970, other sites 0.845.
  - The configuration step mostly costs coverage, the phase choice costs little, and most errors come from the Volt-VAr step (197).
  - In wrong-phase months the voltage-matched phase never has an assessable fit, and 71 of the 100 are at `phase_disagrees`
    sites, so the voltage reference may be what is wrong (not verified).
  - Routing by the site-vote phase changes nothing (0.956).
- **FN and FP plots.**
  - FN: 95 at 27 sites; the top 5 sites hold 44.2%. 68 are too shallow, 27 too steep.
  - The FN plots show three patterns: (1) slopes within about 0.1 %/V of a band edge; (2) household reactive load on the
    tested phase, with the gate dropping intervals; (3) phantom export (below).
  - FP: 41 at 15 sites. The PV circuit's median slope is −0.743 %/V: the inverter barely responds, but the net-meter
    slope lands in the band.
  - Some PV-pass months have Q-V clouds offset from the required curve (e.g. absorption starting near 246 V). The test
    judges the slope only.
- **Phantom export.** 49 site-months (1.1%) at 17 sites have more than 10% of their fitted net-meter intervals with no PV
  output. Agreement there is 0.755 (0.973 for the rest), and they hold 10 of the 95 FN (10.5%) and no TP.
  - The problem is rare but concentrated in errors.
  - A daylight or PV-shape filter is a candidate fix. It would have to be chosen on 2024, reported on 2025, and checked
    on the single-phase method too, which uses the same baseline and gate.
- **Selection bias (7).**
  - C gives only 42 assessable site-months at 9 sites (accuracy 0.952, interval 0.826-1.000), so the check is inconclusive.
  - The end-to-end grade of the non-clean sites (0.845) is the better evidence that the excluded sites are harder.
  - Low PV output months are 2.2% of `month_ok` site-months; the 92 assessable ones all agree.
- **Missing paired circuit (8).** The circuit is absent from `ami_extract` itself: counts are identical in the three tables
  every month. 81 sites (30 lose 2024, 51 lose 2025), 929 incomplete site-months.
  - In their intact year the affected sites validate at least as well (2024: 0.995 vs 0.976; 2025: 0.966 vs 0.957).
  - The sites that lost 2024 pass less often by the reference in 2025 (10.1% vs 20.8%), so dropping their missing year
    could shift fleet conformance rates.
  - The Athena check is pending.
- **Other-phase effect (9).** Paired-phase night baseline: median 1.75% (P) and −2.33% (Q) of rating, vs 4.94% and −3.78% for
  a single-phase house. Median |β_e| 0.085 vs 0.108 %/V, median sd(e) 1.42 vs 1.78, corr(β_e, slope error) 0.931 vs 0.941.
  The error mechanism is the same and slightly smaller: the other phases add no error.